In [ ]:
# Notebook 01: L1 Dummy Data Generation
# Produces L1_rolling.csv and L1_rolling.json - 359 rows, Jan-04 to Dec-28
 
# Cell 1: Imports and global constants
# All window, boundary, and closure settings live here - change here only

import pandas as pd
import numpy as np
import json
from scipy.stats import truncnorm

In [ ]:
# Cell 2: Date spine and seasonal signal
# Phase offset places dummy peak in spring - distinct from real December peak

SEED       = 77
RNG        = np.random.default_rng(SEED)
WINDOW     = 7
HALF       = WINDOW // 2    # 3 rows excluded at each boundary end
START      = "2023-01-01"
END        = "2023-12-31"
DATA_START = "2023-01-04"   # first valid smoothed date after boundary exclusion
DATA_END   = "2023-12-28"   # last valid smoothed date before boundary exclusion
ZERO_DATES = ["2023-12-25"] # set to zero before rolling - store closure day
NULL_DATES = []             # set to NaN before rolling - fully excluded from windows
 
print("Constants loaded")
print(f"Window={WINDOW}, boundary={HALF} rows each end")
print(f"Output range: {DATA_START} to {DATA_END}")

dates        = pd.date_range(START, END, freq="D")
N            = len(dates)
day_idx      = np.arange(N)
 
seasonal     = 1.0 + 0.18 * np.sin((day_idx / N) * 2 * np.pi + np.pi * 0.3)
is_weekend   = pd.Series(dates).dt.dayofweek.isin([5, 6]).astype(float).values
weekend_bump = 1.0 + 0.08 * is_weekend
 
print(f"Date spine: {dates[0].date()} to {dates[-1].date()}, {N} days")

In [ ]:
# Cell 3: Generate raw daily values for all metrics
# Truncated normal keeps each column within realistic physical bounds

def trunc(mean, std, low, high, size):
    a = (low  - mean) / std
    b = (high - mean) / std
    return truncnorm.rvs(a, b, loc=mean, scale=std, size=size, random_state=SEED)
 
total_GHGE_SF = trunc(820_000,      95_000,     500_000,    1_100_000,    N) * seasonal * weekend_bump
total_LU_SF   = trunc(2_100_000,   480_000,     900_000,    3_400_000,    N) * seasonal * weekend_bump
total_WU_SF   = trunc(140_000_000, 28_000_000,  60_000_000, 200_000_000,  N) * seasonal * weekend_bump
 
avg_GHGE_perkg = trunc(5.2,   1.1,  2.0,  9.0,  N)
avg_LU_perkg   = trunc(11.8,  2.6,  4.0,  22.0, N)
avg_WU_perkg   = trunc(270.0, 55.0, 80.0, 500.0, N)
 
GHGE_MAX = 95.0
LU_MAX   = 380.0
WU_MAX   = 5_500.0
 
total_GHGE_scaled = total_GHGE_SF / GHGE_MAX
total_LU_scaled   = total_LU_SF   / LU_MAX
total_WU_scaled   = total_WU_SF   / WU_MAX
 
avg_composite_SF             = (total_GHGE_scaled + total_LU_scaled + total_WU_scaled) / 3.0
avg_composite_SF_norm        = avg_composite_SF / avg_composite_SF.max()
 
GHGE_perkg_norm              = avg_GHGE_perkg / avg_GHGE_perkg.max()
LU_perkg_norm                = avg_LU_perkg   / avg_LU_perkg.max()
WU_perkg_norm                = avg_WU_perkg   / avg_WU_perkg.max()
avg_composite_intensity_norm = (GHGE_perkg_norm + LU_perkg_norm + WU_perkg_norm) / 3.0
 
for name, arr in [
    ("total_GHGE_SF",                total_GHGE_SF),
    ("total_LU_SF",                  total_LU_SF),
    ("total_WU_SF",                  total_WU_SF),
    ("avg_GHGE_perkg",               avg_GHGE_perkg),
    ("avg_composite_SF_norm",        avg_composite_SF_norm),
    ("avg_composite_intensity_norm", avg_composite_intensity_norm),
]:
    print(f"  {name:<35} min={arr.min():.4f}  max={arr.max():.4f}")

In [ ]:
# Cell 4: Build raw L1 DataFrame and apply closure masks
# Masks applied before rolling so closure days contribute correctly to windows

ALL_SRC_COLS = [
    "total_GHGE_SF", "total_LU_SF", "total_WU_SF",
    "avg_GHGE_perkg", "avg_LU_perkg", "avg_WU_perkg",
    "avg_composite_SF_norm", "avg_composite_intensity_norm",
]
 
L1 = pd.DataFrame({
    "date"                         : [d.strftime("%Y-%m-%d") for d in dates],
    "total_GHGE_SF"                : total_GHGE_SF.round(2),
    "total_LU_SF"                  : total_LU_SF.round(2),
    "total_WU_SF"                  : total_WU_SF.round(2),
    "avg_GHGE_perkg"               : avg_GHGE_perkg.round(6),
    "avg_LU_perkg"                 : avg_LU_perkg.round(6),
    "avg_WU_perkg"                 : avg_WU_perkg.round(6),
    "avg_composite_SF_norm"        : avg_composite_SF_norm.round(6),
    "avg_composite_intensity_norm" : avg_composite_intensity_norm.round(6),
})
 
for d in ZERO_DATES:
    mask = L1["date"] == d
    L1.loc[mask, ALL_SRC_COLS] = 0.0
    print(f"Zero mask applied: {d} ({int(mask.sum())} rows)")
 
for d in NULL_DATES:
    mask = L1["date"] == d
    L1.loc[mask, ALL_SRC_COLS] = np.nan
    print(f"Null mask applied: {d} ({int(mask.sum())} rows)")
 
print(f"L1 shape: {L1.shape}")

In [ ]:
# Cell 5: Compute 7-day rolling means for all 8 output columns
# Rolling applied first then boundary masking - do not reverse this order

ROLLING_MAP = {
    "roll7_GHGE_SF"                  : "total_GHGE_SF",
    "roll7_LU_SF"                    : "total_LU_SF",
    "roll7_WU_SF"                    : "total_WU_SF",
    "roll7_composite_norm"           : "avg_composite_SF_norm",
    "roll7_GHGE_perkg"               : "avg_GHGE_perkg",
    "roll7_LU_perkg"                 : "avg_LU_perkg",
    "roll7_WU_perkg"                 : "avg_WU_perkg",
    "roll7_composite_intensity_norm" : "avg_composite_intensity_norm",
}
 
def rolling_clean(series):
    # Rolling mean with strict full-window requirement
    rolled = series.rolling(window=WINDOW, center=True, min_periods=WINDOW).mean()
    # Boundary masking - first and last HALF rows set to NaN
    rolled.iloc[:HALF]  = np.nan
    rolled.iloc[-HALF:] = np.nan
    return rolled
 
for out_col, src_col in ROLLING_MAP.items():
    L1[out_col] = rolling_clean(L1[src_col]).round(6)
 
print("Rolling column verification (all must show n=359):")
ref_n = L1["roll7_GHGE_SF"].notna().sum()
for col in ROLLING_MAP:
    n     = L1[col].notna().sum()
    first = L1.loc[L1[col].notna(), "date"].min()
    last  = L1.loc[L1[col].notna(), "date"].max()
    flag  = "OK" if n == ref_n else "MISMATCH"
    print(f"  {col:<40} n={n}  {first} -> {last}  {flag}")

In [ ]:
# Cell 6: Filter to valid date range and export CSV and JSON
# Only DATA_START to DATA_END rows exported - no null boundary rows included

EXPORT_COLS = [
    "date",
    "roll7_GHGE_SF",
    "roll7_LU_SF",
    "roll7_WU_SF",
    "roll7_composite_norm",
    "roll7_GHGE_perkg",
    "roll7_LU_perkg",
    "roll7_WU_perkg",
    "roll7_composite_intensity_norm",
]

# Full year export: Jan-01 to Dec-31
# Rolling columns are null for boundary dates - chart skips null rows naturally
L1_export = L1[EXPORT_COLS].copy().reset_index(drop=True)
 
null_counts = L1_export.drop(columns=["date"]).isnull().sum()
print("Null counts after filter (all must be 0):")
for col, n in null_counts.items():
    print(f"  {col:<40} {n}")
    
print(f"Rows: {len(L1_export)} (expected 365)")
print(f"Null boundary rows (Jan 1-3, Dec 29-31): "
      f"{L1_export['roll7_GHGE_SF'].isnull().sum()} "
      f"(expected 6)")

print(f"Range: {L1_export['date'].min()} -> {L1_export['date'].max()}")
 
# L1_export.to_csv("./data/L1_rolling_V2.csv", index=False)
# print("Saved: ./data/L1_rolling_V2.csv")
 
L1_export.to_json("./data/L1_rolling_V2.json", orient="records", indent=2)
print("Saved: ./data/L1_rolling_V2.json")
 
with open("./data/L1_rolling_V2.json") as f:
    check = json.load(f)
print(f"JSON records: {len(check)}")
print(f"First: {check[0]['date']}  roll7_GHGE_SF={check[0]['roll7_GHGE_SF']}")
print(f"Last:  {check[-1]['date']}  roll7_GHGE_SF={check[-1]['roll7_GHGE_SF']}")